# 14-Class Model Selector Training (Colab)

Use this notebook to train one model at a time by changing `MODEL_NAME` in the config cell.

Supported models:
- `cnn`
- `lstm`
- `bilstm`
- `ornn`
- `cnn_ornn`
- `siao_cnn_ogru` (your proposed full pipeline)

In [ ]:
import subprocess

print('=' * 55)
print(' ENVIRONMENT CHECK')
print('=' * 55)

try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
        capture_output=True, text=True, check=True
    )
    print('GPU :', result.stdout.strip())
except Exception:
    print('GPU : No GPU detected. Switch runtime to T4/A100.')

with open('/proc/meminfo') as f:
    lines = f.readlines()
mem = {l.split(':')[0]: l.split(':')[1].strip() for l in lines}
total_gb = int(mem['MemTotal'].split()[0]) / 1024 / 1024
avail_gb = int(mem['MemAvailable'].split()[0]) / 1024 / 1024
print(f'RAM : {total_gb:.1f} GB total, {avail_gb:.1f} GB available')
print('=' * 55)

In [ ]:
# 1) Clone repo and install requirements
# Replace REPO_URL with your repository URL.
REPO_URL = 'https://github.com/<your-username>/siao-cnn-ogru.git'

import os

if not os.path.exists('/content/siao-cnn-ogru'):
    !git clone {REPO_URL} /content/siao-cnn-ogru

%cd /content/siao-cnn-ogru
!pip install -q -r requirements.txt

In [ ]:
# 2) Upload Operation_csv_data zip and extract to data/
import glob
import os
import zipfile
from google.colab import files

print('Select your Operation_csv_data zip file:')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

os.makedirs('data', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('data/')

op_candidates = glob.glob('data/**/Operation_csv_data', recursive=True)
DATA_DIR = op_candidates[0] if op_candidates else 'data/Operation_csv_data'
print('Using DATA_DIR =', DATA_DIR)
print('Classes found:', sorted(os.listdir(DATA_DIR)))

In [ ]:
# 3) Configure one model here
from src.siao_cnn_ogru.data.class_metadata import RESEARCH_CLASS_CODES_14
from src.siao_cnn_ogru.training import list_supported_models

print('Supported models:', list_supported_models())

MODEL_NAME = 'cnn'  # change to: cnn, lstm, bilstm, ornn, cnn_ornn, siao_cnn_ogru

CONFIG = {
    'model': MODEL_NAME,
    'data_dir': DATA_DIR,
    'active_class_codes': RESEARCH_CLASS_CODES_14,
    'max_timesteps': 300,
    'window_size': 100,
    'stride': 25,
    'n_folds': 10,
    'epochs': 100,
    'batch_size': 128,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'hidden_size': 128,
    'num_layers': 2,
    'dropout': 0.3,
    'use_class_weights': True,
    'label_smoothing': 0.02,
    'use_smote': False,
    'patience': 15,
    'save_dir': '/content/siao-cnn-ogru/results',
}

# Proposed model-specific knobs (used only when model='siao_cnn_ogru')
CONFIG.update({
    'siao_pop_size': 25,
    'siao_max_iter': 40,
    'bp_epochs': 100,
    'bp_lr': 0.00157,
    'fc_dropout': 0.164,
})

CONFIG

In [ ]:
# 4) Train selected model
from train_model_selector import run_from_config

results = run_from_config(CONFIG)

print('\n' + '=' * 65)
print('Model:', results.get('model', MODEL_NAME))
print(f"Average Accuracy: {results['avg_accuracy']*100:.2f}%")
print(f"Std Accuracy: {results['std_accuracy']*100:.2f}%")
print(f"Average Macro-F1: {results['avg_macro_f1']*100:.2f}%")
print('Summary JSON:', results.get('summary_path', 'N/A'))
print('Reliability JSON:', results.get('reliability_summary_path', 'N/A'))
print('=' * 65)

In [ ]:
# 5) Quick reliability view
rel = results.get('reliability', {})
cls = rel.get('classification', {})
dyn = rel.get('reliability', {})

print('Accuracy      :', round(cls.get('accuracy', 0.0), 4))
print('Macro Precision:', round(cls.get('macro_precision', 0.0), 4))
print('Macro Recall   :', round(cls.get('macro_recall', 0.0), 4))
print('Macro F1       :', round(cls.get('macro_f1', 0.0), 4))
print('Final MTTF     :', round(dyn.get('final_mttf_pred', 0.0), 4))
print('Final Reliability:', round(dyn.get('final_reliability_pred', 0.0), 4))
print('Curve Scores:', dyn.get('curve_scores', {}))

In [ ]:
# 6) Zip and download results
import shutil
from google.colab import files

archive_base = f"/content/{MODEL_NAME}_14class_results"
zip_path = shutil.make_archive(archive_base, 'zip', '/content/siao-cnn-ogru/results')
print('Created:', zip_path)
files.download(zip_path)